In [625]:
import pandas as pd
import jieba
import re
import wordcloud
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer  
import numpy as np
from sklearn.feature_extraction.text import TfidfTransformer  
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation
import mglearn

In [626]:
data_2018=pd.read_excel("./low_2018.xlsx")

In [627]:
data_2018.head()

,mid,日期,内容,转发数,评论数,点赞数,来自,博主,uid,是否已认证,认证内容
0,4323349901735420,2018-12-31 18:00,输卵管具有运送精子、摄取卵子及把受精卵运送到子宫腔的重要作用， 一旦发生堵塞、积水、不通等，...,NaN,NaN,NaN,深圳怡康妇产医院V,微博 weibo.com,6362406344,True,深圳怡康妇产医院官方微博
1,4323327252420440,2018-12-31 16:30,【怡康喜报】二胎政策放开后，张女士全家特别想再要一个孩子。取环后，夫妻俩积极备孕，多次尝试，...,NaN,NaN,NaN,深圳怡康妇产医院V,微博 weibo.com,6362406344,True,深圳怡康妇产医院官方微博
2,4323305915956700,2018-12-31 15:05,#不孕不育#宜正堂 治疗不孕不育20年 ，不孕不育的专家 联系电话1662116...,NaN,NaN,NaN,宜正堂,vivo AI智慧拍照X21,6906732347,False,NaN
3,4323214005703270,2018-12-31 09:00,为什么怀孕了，建议大家及时去做早孕检查。是因为大家在做试纸检测的时候，所操作方法和试纸厂家存...,NaN,NaN,NaN,深圳怡康妇产医院V,微博 weibo.com,6362406344,True,深圳怡康妇产医院官方微博
4,4323041046146520,2018-12-30 21:32,#南宁 不孕不育 试管移植失败#宜正堂 治疗不孕不育的专家 联系电话166211...,NaN,NaN,NaN,宜正堂,vivo AI智慧拍照X21,6906732347,False,NaN


In [568]:
data_2018.shape

(1960, 11)

In [569]:
data_2018=data_2018[["内容"]]

In [570]:
data_2018.head()

,内容
0,输卵管具有运送精子、摄取卵子及把受精卵运送到子宫腔的重要作用， 一旦发生堵塞、积水、不通等，...
1,【怡康喜报】二胎政策放开后，张女士全家特别想再要一个孩子。取环后，夫妻俩积极备孕，多次尝试，...
2,#不孕不育#宜正堂 治疗不孕不育20年 ，不孕不育的专家 联系电话1662116...
3,为什么怀孕了，建议大家及时去做早孕检查。是因为大家在做试纸检测的时候，所操作方法和试纸厂家存...
4,#南宁 不孕不育 试管移植失败#宜正堂 治疗不孕不育的专家 联系电话166211...


In [571]:
data_2018=data_2018.drop_duplicates()

In [572]:
data_2018.head()

,内容
0,输卵管具有运送精子、摄取卵子及把受精卵运送到子宫腔的重要作用， 一旦发生堵塞、积水、不通等，...
1,【怡康喜报】二胎政策放开后，张女士全家特别想再要一个孩子。取环后，夫妻俩积极备孕，多次尝试，...
2,#不孕不育#宜正堂 治疗不孕不育20年 ，不孕不育的专家 联系电话1662116...
3,为什么怀孕了，建议大家及时去做早孕检查。是因为大家在做试纸检测的时候，所操作方法和试纸厂家存...
4,#南宁 不孕不育 试管移植失败#宜正堂 治疗不孕不育的专家 联系电话166211...


In [573]:
def remove_hashtags(text):
    # 使用正则表达式匹配两个#之间的内容，并替换为空字符串
    cleaned_text = re.sub(r'#(.*?)#', '', text)
    return cleaned_text

In [574]:
#删除话题标志
data_2018["内容"]=data_2018["内容"].apply(lambda x:remove_hashtags(x))

In [575]:
#用大模型判断是否是不孕不育话题
import requests
def send_post(url,data):
    try:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            #             print("POST request was successful!")
            #             print("Response data:")
            return response.json()
        else:
            print(f"POST request failed with status code: {response.status_code}")
    except requests.exceptions.RequestException as e:
        print(f"Error occurred: {e}")
url="http://180.76.49.225:7861/chat"

In [576]:
realt_flag=[]

In [ ]:
for cont in data_2018["内容"].values:
    data={"question":"请帮我判断以下文本是否关于不孕不育、生育或者怀孕的话题，如果是，请直接输出一个字是，否则直接输出不是，务必不要返回其他内容。文本是："+cont}
    realt_flag.append(send_post(url,data)["answer"])

In [555]:
data_2018["relat"]=realt_flag

In [556]:
data_2018=data_2018[data_2018["relat"].apply(lambda x:x=="是")]

In [560]:
data_2018.shape

(1602, 2)

In [561]:
data_2018.to_excel("./2018_rel.xlsx")

In [710]:
data_2018=pd.read_excel("./2018_rel.xlsx")

In [711]:
#替换名称
cn_city=["四川省","四川","南岳","上海","广州","深圳","东莞","重庆","北京","合肥","郑州","天津","无锡","南宁","忻州","长沙","南宁",
         "南王","漕河镇","黄冈","湖南","十安","湖南","十安","南京","信阳","沈阳","台湾","汕头","广东","河北","郴州","邯郸","海口"]
out_city=["法国","泰国","澳洲","乌克兰","曼谷","马来西亚","英国政府","美国","欧洲","阿德莱德","俄罗斯"]
person=["顾岩","田纪香","王丽敏","王铁梅","铁梅","庹有烈","匡淑杰","田平君","田平","医羽","爱问","张潇潇","李媛","崔娟","郑芬芬"]
hospi=["怡康","九院","百佳","宝来佳运","恒健","济爱","宜正堂","卫人","仁和","北辰","朝阳医院","衍宗堂","和睦家","仁爱医院","华博"]

In [712]:
def replace(ss):
    for c in cn_city:
        ss=ss.replace(c,"国内")
    for c in out_city:
        ss=ss.replace(c,"国外")
    for c in person:
        ss=ss.replace(c,"医生")
    for c in hospi:
        ss=ss.replace(c,"医院")
    return ss

In [713]:
data_2018["内容"]=data_2018["内容"].apply(lambda x:replace(x))

In [714]:
#只保留中文
data_2018["content2"]=data_2018["内容"].apply(lambda x:re.sub(r'[^\u4e00-\u9fa5]','',str(x)))

In [715]:
jieba.add_word("公众号")

In [716]:
#结巴分词
data_2018["words"]=data_2018["content2"].apply(lambda x:jieba.lcut(x))

In [717]:
data_2018.head()

,Unnamed: 0,内容,relat,content2,words
0,0,输卵管具有运送精子、摄取卵子及把受精卵运送到子宫腔的重要作用， 一旦发生堵塞、积水、不通等，...,是,输卵管具有运送精子摄取卵子及把受精卵运送到子宫腔的重要作用一旦发生堵塞积水不通等需要及时治疗...,"[输卵管, 具有, 运送, 精子, 摄取, 卵子, 及, 把, 受精卵, 运送, 到, 子宫..."
1,1,【医院喜报】二胎政策放开后，张女士全家特别想再要一个孩子。取环后，夫妻俩积极备孕，多次尝试，...,是,医院喜报二胎政策放开后张女士全家特别想再要一个孩子取环后夫妻俩积极备孕多次尝试可一年多过去了...,"[医院, 喜报, 二胎, 政策, 放开, 后, 张女士, 全家, 特别, 想, 再, 要, ..."
2,2,医院 治疗不孕不育20年 ，不孕不育的专家 联系电话16621161987微信同...,是,医院治疗不孕不育年不孕不育的专家联系电话微信同号国内武鸣县城厢镇,"[医院, 治疗, 不孕, 不育, 年, 不孕, 不育, 的, 专家, 联系电话, 微信, 同..."
3,3,为什么怀孕了，建议大家及时去做早孕检查。是因为大家在做试纸检测的时候，所操作方法和试纸厂家存...,是,为什么怀孕了建议大家及时去做早孕检查是因为大家在做试纸检测的时候所操作方法和试纸厂家存放时间...,"[为什么, 怀孕, 了, 建议, 大家, 及时, 去, 做, 早孕, 检查, 是因为, 大家..."
4,4,医院 治疗不孕不育的专家 联系电话16621161987 2国内·那洪街区 ​,是,医院治疗不孕不育的专家联系电话国内那洪街区,"[医院, 治疗, 不孕, 不育, 的, 专家, 联系电话, 国内, 那洪, 街区]"


In [718]:
#读停用词
StopDictionary_path = './stop_words.txt'  #stop_words_2018
stop_words = []

In [719]:
pd.DataFrame(stop_words).to_csv("./2018_stopword.csv")

In [720]:
# 加载停用词
def get_stopwords(filePath):
    global stop_words
    f = open(filePath, 'r', encoding='utf8', errors='ignore')
    line = f.readline()
    while line:
        stop_words.append(line.strip())
        line = f.readline()
    f.close()

In [721]:
get_stopwords(StopDictionary_path)

In [722]:
#去掉数据中的停用词
def drop_stop_words(arr):
    word_clean=[]
    for w in arr:
        if len(w)>=2 and w not in stop_words:
            word_clean.append(w)
    return word_clean

In [723]:
data_2018["words_clean"]=data_2018["words"].apply(lambda x:drop_stop_words(x))

In [724]:
data_2018.head()

,Unnamed: 0,内容,relat,content2,words,words_clean
0,0,输卵管具有运送精子、摄取卵子及把受精卵运送到子宫腔的重要作用， 一旦发生堵塞、积水、不通等，...,是,输卵管具有运送精子摄取卵子及把受精卵运送到子宫腔的重要作用一旦发生堵塞积水不通等需要及时治疗...,"[输卵管, 具有, 运送, 精子, 摄取, 卵子, 及, 把, 受精卵, 运送, 到, 子宫...","[输卵管, 运送, 精子, 摄取, 卵子, 受精卵, 运送, 子宫腔, 作用, 发生, 堵塞..."
1,1,【医院喜报】二胎政策放开后，张女士全家特别想再要一个孩子。取环后，夫妻俩积极备孕，多次尝试，...,是,医院喜报二胎政策放开后张女士全家特别想再要一个孩子取环后夫妻俩积极备孕多次尝试可一年多过去了...,"[医院, 喜报, 二胎, 政策, 放开, 后, 张女士, 全家, 特别, 想, 再, 要, ...","[医院, 喜报, 二胎, 政策, 放开, 全家, 特别, 孩子, 取环后, 夫妻俩, 备孕,..."
2,2,医院 治疗不孕不育20年 ，不孕不育的专家 联系电话16621161987微信同...,是,医院治疗不孕不育年不孕不育的专家联系电话微信同号国内武鸣县城厢镇,"[医院, 治疗, 不孕, 不育, 年, 不孕, 不育, 的, 专家, 联系电话, 微信, 同...","[医院, 治疗, 不孕, 不育, 不孕, 不育, 专家, 联系电话, 同号, 国内, 武鸣县..."
3,3,为什么怀孕了，建议大家及时去做早孕检查。是因为大家在做试纸检测的时候，所操作方法和试纸厂家存...,是,为什么怀孕了建议大家及时去做早孕检查是因为大家在做试纸检测的时候所操作方法和试纸厂家存放时间...,"[为什么, 怀孕, 了, 建议, 大家, 及时, 去, 做, 早孕, 检查, 是因为, 大家...","[怀孕, 建议, 早孕, 检查, 试纸, 检测, 操作方法, 试纸, 厂家, 存放, 时间,..."
4,4,医院 治疗不孕不育的专家 联系电话16621161987 2国内·那洪街区 ​,是,医院治疗不孕不育的专家联系电话国内那洪街区,"[医院, 治疗, 不孕, 不育, 的, 专家, 联系电话, 国内, 那洪, 街区]","[医院, 治疗, 不孕, 不育, 专家, 联系电话, 国内, 街区]"


In [725]:
data_2018["words_str"]=data_2018["words_clean"].apply(lambda x:" ".join(x))

In [726]:
#将文本中的词语转换为词频矩阵  
vectorizer = CountVectorizer()  
#计算个词语出现的次数  
X = vectorizer.fit_transform(data_2018["words_str"])  
#获取词袋中所有文本关键词  
word = vectorizer.get_feature_names()  
print (word)
#查看词频结果  
print(X.toarray())

['一万', '一万三千', '一万元', '一万次', '一下子', '一不小心', '一不留神', '一丝', '一两个', '一两天', '一两杯', '一两次', '一个个', '一个多月', '一个月', '一举', '一久', '一事', '一事无成', '一二十年', '一些第三', '一人', '一代', '一件', '一份', '一会', '一会儿', '一位', '一体', '一侧', '一儿', '一共', '一击', '一分钟', '一切正常', '一切顺利', '一到', '一刻', '一千', '一千多', '一半', '一半左右', '一双', '一取', '一口', '一口气', '一句', '一只', '一同', '一名', '一向', '一周', '一味', '一喷灵', '一回', '一团', '一圈', '一在', '一场', '一坐', '一块', '一堆', '一多囊', '一大', '一大堆', '一大早', '一套', '一女', '一子', '一孕', '一孩', '一定量', '一家', '一对', '一对一', '一对对', '一對雙', '一小', '一小部分', '一层', '一岁', '一年', '一年半载', '一年四季', '一度', '一张', '一微信', '一成不变', '一批', '一拖再拖', '一整天', '一文', '一方', '一无所获', '一日', '一日三餐', '一早', '一是', '一晒锦泰', '一月', '一有', '一朵', '一村', '一条', '一杨', '一杯', '一枚', '一架', '一查', '一根', '一棵树上吊死', '一次性', '一次次', '一步', '一步一个脚印', '一步到位', '一步步', '一段', '一段时间', '一水', '一波', '一波用', '一派', '一流', '一测', '一滴水', '一点', '一点很多', '一点点', '一父', '一环', '一班', '一生', '一男一女', '一百多万', '一百多个', '一盒', '一盘', '一看', '一知半解', '一种', '一站', '一端', '一竿子', '一笔', '一篇', '一类', '一粒', '一精卵', '一系列', '一级', '一线

In [727]:
#词的维度
print(len(word))

11474


In [728]:
#文本表述维度
X.toarray().shape

(1602, 11474)

In [729]:
#类调用  
transformer = TfidfTransformer()  
#将词频矩阵X统计成TF-IDF值  
tfidf = transformer.fit_transform(X)  
#查看数据结构 tfidf[i][j]表示i类文本中的tf-idf权重  
print(tfidf.toarray()  )

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [730]:
#查看第一个文本的词的tfidf值
for i in tfidf.toarray()[0]:
    if i!=0:
        print(i)

0.2452292301017844
0.1808819890312192
0.1443937027621794
0.1664420271640152
0.18997764335496417
0.18268119995739526
0.2239229493634262
0.23055351905217888
0.10233053527585158
0.2672303050371393
0.119097364200075
0.13787971785566366
0.2383426461514016
0.2209435544637982
0.1401830303350684
0.1319985522805218
0.5748449081191365
0.3124229348092836


In [734]:
k=3

In [735]:
#LDA主题建模
#设置主题数量，学习方式，超参数α，β取默认值。
lda = LatentDirichletAllocation(n_components=k, learning_method='batch', max_iter=50, random_state=3)
lda_topics = lda.fit_transform(tfidf.toarray())

In [736]:
#对主题词做降序排列
sorting = np.argsort(lda.components_, axis=1)[:, ::-1]
feature_names = np.array(vectorizer.get_feature_names())

mglearn.tools.print_topics(topics=range(k), feature_names=feature_names,
                           sorting=sorting, topics_per_chunk=5, n_words=20)

topic 0       topic 1       topic 2       
--------      --------      --------      
预约            主任            试管婴儿          
国内            怀孕            国外            
优惠            医院            不孕            
意想不到          恭喜            医院            
公众号           报喜            女性            
街区            医生            试管            
教授            输卵管           子宫            
宝典            调理            不育            
最全            宝宝            精子            
门诊            国内            胚胎            
关注            怀上            检查            
会诊            妇产医院          怀孕            
联系电话          就诊            国内            
号码            成功            输卵管           
身份证           中西医           影响            
抓紧时间          治疗            卵巢            
妇产医院          好孕            治疗            
老店            艾灸            内膜            
东院            腹腔镜           移植            
同号            确诊            原因            




In [602]:
for k in range(3,7):
    print("============"+str(k)+"============")
    #LDA主题建模
    #设置主题数量，学习方式，超参数α，β取默认值。
    lda = LatentDirichletAllocation(n_components=k, learning_method='batch', max_iter=50, random_state=0)
    lda_topics = lda.fit_transform(tfidf.toarray())
    #对主题词做降序排列
    sorting = np.argsort(lda.components_, axis=1)[:, ::-1]
    feature_names = np.array(vectorizer.get_feature_names())

    mglearn.tools.print_topics(topics=range(k), feature_names=feature_names,
                               sorting=sorting, topics_per_chunk=5, n_words=20)

============3============
topic 0       topic 1       topic 2       
--------      --------      --------      
爸爸            试管婴儿          医院            
缩阴            国外            主任            
艾灸            不孕            国内            
懷上            医院            医生            
男寶寶           女性            怀孕            
高峰论坛          试管            恭喜            
想当            子宫            报喜            
老婆            不育            调理            
卧室            精子            宝宝            
生二胎           胚胎            输卵管           
准备就绪          怀孕            怀上            
具体表现          检查            就诊            
忌讳            输卵管           预约            
感动            影响            国际            
听说            卵巢            妇产医院          
饮料            治疗            成功            
两盒            内膜            中西医           
动力            移植            治疗            
老公            国内            生殖            
喜悦            原因            好孕            


============4============
